## Create a file with word distributions and intruders

In [1]:
import pandas as pd
import numpy as np
import random

In [2]:
HOME_DIR = "Distributions-Results"
DATA_SET = "20NewsGroup"
MODEL = "prior_GMM"

In [3]:
import os
os.chdir(path="..")
cwd = os.getcwd()
cwd

'/home/dsi/ishonta/TM'

In [4]:
def get_top_words_for_topics(topic_word_df, top_n=8):
    top_words_by_topic = {}
    for topic_id in topic_word_df['Topic']:
        # Get the row corresponding to the current topic (excluding the 'Topic' column)
        topic_row = topic_word_df.loc[topic_word_df['Topic'] == topic_id].drop(columns='Topic').iloc[0]
        # Sort the row values in descending order and get the top N columns (words)
        top_words = topic_row.sort_values(ascending=False).head(top_n)
        
        # Collect the top N words (column names)
        top_words_by_topic[topic_id] = top_words.index.tolist()
    
    return top_words_by_topic

In [130]:
def get_intruder_words(topic_word_df, top_k=8, num_lowest_words=10):
    """
    Identify intruder words for each topic.
    - Finds low-probability words in each topic.
    - Checks if these words are among the top-k words in another topic.
    - If a low-probability word is not found in any top-k list, a random high-probability word from another topic is selected.
    """
    # Get top-k words for all topics
    top_words_by_topic = get_top_words_for_topics(topic_word_df, top_n=top_k)
    intruder_words = {}

    for topic_id in topic_word_df['Topic'].unique():
        # Get the current topic row
        topic_row = topic_word_df.loc[topic_word_df['Topic'] == topic_id].drop(columns='Topic').iloc[0]

        # Find the lowest-probability words above zero
        low_prob_words = topic_row[topic_row > 0].sort_values().head(num_lowest_words).index

        intruder_word = None

        # Check if any low-probability word is an intruder
        for word in low_prob_words:
            found_intruder = False

            # Check if the word appears in the top-k list of any other topic
            for other_topic_id, top_k_words in top_words_by_topic.items():
                if other_topic_id != topic_id and word in top_k_words:
                    intruder_word = word
                    found_intruder = True
                    break  # Stop looking for other topics once an intruder is found

            # If the word is not found in any top-k list, select a random word from another topic
            if not found_intruder:
                # Get the list of words not in the current topic's top-k words
                current_topic_top_k = set(top_words_by_topic[topic_id])
                potential_intruders = set()
                for other_topic_id, top_k_words in top_words_by_topic.items():
                    if other_topic_id != topic_id:
                        # Add words from the other topic that are not in the current topic's top-k
                        potential_intruders.update([word for word in top_k_words if word not in current_topic_top_k])

                # Randomly select one word from the potential intruders
                if potential_intruders:
                    intruder_word = random.choice(list(potential_intruders))
                else:
                    print(f"No suitable high-probability word found for topic {topic_id}")

            if intruder_word:
                intruder_words[topic_id] = intruder_word
                break
            else:
                print(f"No intruder found for topic {topic_id}")

    return intruder_words


In [129]:
def insert_intruders_into_top_words(topic_word_df, top_n=8):
    """
    Insert intruder words into the top N words for each topic.
    """
    # Get the top words for each topic
    top_words_by_topic = get_top_words_for_topics(topic_word_df, top_n=top_n)

    # Find the intruder words for each topic
    intruder_words = get_intruder_words(topic_word_df)

    # Insert the intruder word among the top words for each topic
    updated_top_words_by_topic = {}
    for topic_id, top_words in top_words_by_topic.items():
        intruder = intruder_words[topic_id]
        if intruder not in top_words:
            # Insert the intruder word (ensure the list remains unique)
            updated_top_words = top_words + [intruder]
            random.shuffle(updated_top_words)
            print(f"intruder: {intruder}")
            print(f"top words: {top_words}")
        else:
            print("GOT HERE")
            print(f"intruder: {intruder}")
            print(f"top words: {top_words}")
            updated_top_words = top_words
        updated_top_words_by_topic[topic_id] = updated_top_words

    return updated_top_words_by_topic, intruder_words

In [137]:
dir_path = f"helper/{DATA_SET}"

# Create directory if it doesn't exist
if not os.path.exists(dir_path):
    os.makedirs(dir_path)

for MODEL in ["BERTopic", "lda", "prior_GMM", "prior_ScaSE"]:
    df = pd.read_csv(f"{HOME_DIR}/{DATA_SET}/{MODEL}_topic_word_distribution.csv")
    df = df.round(5)
    updated_top_words_by_topic, intruder_words = insert_intruders_into_top_words(df)
    updated_df = pd.DataFrame(updated_top_words_by_topic)
    intruders_df = pd.DataFrame(intruder_words, index=[0])
    updated_df.to_csv(f"{dir_path}/{MODEL}_intruder_check.csv")
    intruders_df.to_csv(f"{dir_path}/{MODEL}_the_intruders.csv")

intruder: motherboard
top words: ['game', 'team', 'player', 'play', 'season', 'hockey', 'league', 'baseball']
intruder: word
top words: ['doctor', 'patient', 'disease', 'treatment', 'medical', 'cause', 'effect', 'food']
intruder: label
top words: ['space', 'orbit', 'nasa', 'launch', 'mission', 'shuttle', 'moon', 'earth']
intruder: paint
top words: ['woof', 'gehrig', 'cheek', 'headline', 'outrage', 'trip', 'three', 'though']
intruder: intel
top words: ['israel', 'israeli', 'arab', 'palestinian', 'jewish', 'peace', 'muslim', 'bosnia']
intruder: motif
top words: ['window', 'widget', 'display', 'problem', 'program', 'application', 'file', 'manager']
intruder: projector
top words: ['circuit', 'power', 'input', 'voltage', 'audio', 'stereo', 'output', 'radio']
intruder: diamond
top words: ['subscribe', 'test', 'thank', 'ditto', 'loser', 'please', 'retard', 'think']
intruder: secular
top words: ['armenian', 'turkish', 'serdar', 'turk', 'armenia', 'genocide', 'turkey', 'muslim']
intruder: drive

In [131]:
# MODEL = "prior_ScaSE"
# df = pd.read_csv(f"{HOME_DIR}/{DATA_SET}/{MODEL}_topic_word_distribution.csv")
# df = df.round(5)

# lowp, intruders = insert_intruders_into_top_words(df)


intruder: tissue
top words: ['weapon', 'fire', 'koresh', 'police', 'batf', 'compound', 'agent', 'firearm']
intruder: draft
top words: ['forgot', 'knife', 'pocket', 'coat', 'trademark', 'metropolitan', 'hunting', 'stamp']
intruder: tape
top words: ['surveillance', 'instruction', 'intelligence', 'bullock', 'francisco', 'wednesday', 'african', 'amateur']
intruder: real
top words: ['detector', 'radar', 'formula', 'meter', 'moncton', 'baud', 'gazan', 'contribution']
intruder: satellite
top words: ['typing', 'expose', 'seizure', 'pulse', 'reservation', 'needle', 'additive', 'wavefunction']
intruder: fort
top words: ['state', 'government', 'right', 'private', 'american', 'unit', 'citizen', 'insurance']
intruder: fielder
top words: ['military', 'technology', 'attack', 'maria', 'civilian', 'terrorist', 'zone', 'sector']
intruder: transmit
top words: ['file', 'version', 'software', 'avail', 'data', 'system', 'graphic', 'program']
intruder: private
top words: ['back', 'time', 'right', 'year', 'li